In [24]:
### CREATING INDEX FROM ENSENBL ID
import pandas as pd

def num_to_uint8(n):
    return [(n >> 16) & 0xFF, (n >> 8) & 0xFF, n & 0xFF]

# Load the CSV file
# df = pd.read_csv('raw_data_0.csv')

# Process the comp_name column
def process_comp_name(comp_name):
    gene_part = comp_name.split('_')[0]
    gene_number = int(gene_part.lstrip('ENSG').lstrip('0'))
    return num_to_uint8(gene_number)

# Apply the conversion to each row
# df[['gene_idx0', 'gene_idx1', 'gene_idx2']] = pd.DataFrame(df['comp_name'].apply(process_comp_name).tolist(), index=df.index)

# # Print the head and tail of the data showing only comp_name and gene_idx columns
# print(df[['comp_name', 'gene_idx0', 'gene_idx1', 'gene_idx2']].head())
# print(df[['comp_name', 'gene_idx0', 'gene_idx1', 'gene_idx2']].tail())


In [15]:
### CODE TO VERIFY THE INDEX VALUES OF ABOVE
def num_to_uint8(n):
    return [(n >> 16) & 0xFF, (n >> 8) & 0xFF, n & 0xFF]

def uint8_to_num(arr):
    return (arr[0] << 16) | (arr[1] << 8) | arr[2]

# Example usage:
examples = [3, 286215]
converted = [num_to_uint8(n) for n in examples]
reversed_converted = [uint8_to_num(arr) for arr in converted]

# converted, reversed_converted
print(converted)
print(reversed_converted)

[[0, 0, 3], [4, 94, 7]]
[3, 286215]


In [37]:
###RECOLLECT DUPLICATES
import pandas as pd

duplicates_df = pd.read_csv('duplicates_to_remove.csv')
gene_alignments_df = pd.read_csv('/data/home/mrichte3/RNASeq/gene_alignments5.csv')
matched_df = gene_alignments_df[gene_alignments_df['ensembl_id'].isin(duplicates_df['ensembl_id'])]
matched_df = matched_df.drop(columns=['gene_aligned', 'target_rna', 'target_encoding'])
new_entries = []
for index, row in matched_df.iterrows():
    new_entries.append({'comp_name': f"{row['ensembl_id']}_unmod", 'log2FC': row['log2FC_unmod']})
    new_entries.append({'comp_name': f"{row['ensembl_id']}_amide", 'log2FC': row['log2FC_amide']})
    new_entries.append({'comp_name': f"{row['ensembl_id']}_gna", 'log2FC': row['log2FC_gna']})
expanded_df = pd.DataFrame(new_entries)
expanded_df[['gene_idx0', 'gene_idx1', 'gene_idx2']] = pd.DataFrame(expanded_df['comp_name'].apply(process_comp_name).tolist(), index=expanded_df.index)
print(expanded_df[['comp_name', 'gene_idx0', 'gene_idx1', 'gene_idx2']].head())
print(expanded_df[['comp_name', 'gene_idx0', 'gene_idx1', 'gene_idx2']].tail())
# print(expanded_df.head())
print(expanded_df.shape)
# print(matched_df.head())
# print(matched_df.shape)

raw_data_df = pd.read_csv('raw_data_4.csv')
uint8_columns = [col for col in raw_data_df.columns if col.startswith('uint8_')]
gene_idx_columns = [col for col in raw_data_df.columns if col.startswith('gene_idx')]

amide_data = raw_data_df[raw_data_df['comp_name'] == 'ENSG00000000003_amide'][uint8_columns].values.flatten()
unmod_data = raw_data_df[raw_data_df['comp_name'] == 'ENSG00000000003_unmod'][uint8_columns].values.flatten()
gna_data = raw_data_df[raw_data_df['comp_name'] == 'ENSG00000000003_gna'][uint8_columns].values.flatten()

expanded_df.loc[expanded_df['comp_name'].str.contains('amide'), uint8_columns] = amide_data
expanded_df.loc[expanded_df['comp_name'].str.contains('unmod'), uint8_columns] = unmod_data
expanded_df.loc[expanded_df['comp_name'].str.contains('gna'), uint8_columns] = gna_data

expanded_df = expanded_df[['comp_name'] + gene_idx_columns + uint8_columns + ['log2FC']]
print(expanded_df.head())
print(expanded_df.shape)

raw_data_df = pd.read_csv('raw_data_4.csv')
raw_data_df = raw_data_df.drop(columns=['off_target'])
combined_df = pd.concat([expanded_df, raw_data_df], ignore_index=True)
print(combined_df.head())
print(combined_df.shape)

combined_df.to_csv('raw_data_4_updated.csv', index=False)

               comp_name  gene_idx0  gene_idx1  gene_idx2
0  ENSG00000001497_unmod          0          5        217
1  ENSG00000001497_amide          0          5        217
2    ENSG00000001497_gna          0          5        217
3  ENSG00000003393_unmod          0         13         65
4  ENSG00000003393_amide          0         13         65
                  comp_name  gene_idx0  gene_idx1  gene_idx2
1240  ENSG00000273373_amide          4         43        221
1241    ENSG00000273373_gna          4         43        221
1242  ENSG00000285106_unmod          4         89        178
1243  ENSG00000285106_amide          4         89        178
1244    ENSG00000285106_gna          4         89        178
(1245, 5)


In [5]:
import pandas as pd

data = pd.read_csv('raw_data_1.csv')

rna_columns = [col for col in data.columns if col.startswith('rna_')]
residue_columns = [col for col in data.columns if col not in ['comp_name', 'log2FC', 'off_target'] + rna_columns]

rna_data = data[rna_columns].select_dtypes(include=['number'])
residue_data = data[residue_columns].select_dtypes(include=['number'])

combined_data = pd.concat([rna_data, residue_data], axis=1).values.flatten()

min_value = combined_data.min()
max_value = combined_data.max()
mean_value = combined_data.mean()
std_value = combined_data.std()

print(f"Min: {min_value}, Max: {max_value}, Mean: {mean_value}, Std: {std_value}")


Min: -134.97325, Max: 111.87947, Mean: 37.403512432218285, Std: 24.142460493514417


In [8]:
## GENERATES ALL POSSIBLE 8MER SEQUENCES
import pandas as pd
from collections import Counter
import itertools

def generate_8mer_sequences():
    bases = ['A', 'T', 'C', 'G']
    return [''.join(seq) for seq in itertools.product(bases, repeat=8)]
    
def find_top_8mer_matches(file_path):
    df = pd.read_csv(file_path)
    first_gene_sequence = df.iloc[0]['sequence']
    sequences = generate_8mer_sequences()
    print(len(sequences))
    
    counts = Counter([first_gene_sequence[i:i+8] for i in range(len(first_gene_sequence)-7)])
    top_10_matches = counts.most_common(10)
    
    return top_10_matches

file_path = '/data/home/mrichte3/RNASeq/sequences.csv'
top_10_matches = find_top_8mer_matches(file_path)
print(top_10_matches)


65536
[('AAAAAAAA', 17), ('TTTTTTTT', 15), ('AGAAAATT', 7), ('ATTTTCCT', 7), ('TTTTCCTT', 6), ('TTTAATTT', 6), ('TTTGTTTT', 6), ('AAAAGTTT', 6), ('GAAAAAAA', 6), ('TATTTTCC', 5)]


In [11]:
## GENERATE ALL POSSIBLE 8MER, THEN ATTEMPTS TO FIND FREQUENCIES, BUT DOES NOT WORK CORRECTLY
import pandas as pd
from collections import Counter

def generate_8mer_sequences():
    bases = ['A', 'T', 'C', 'G']
    return [''.join(seq) for seq in itertools.product(bases, repeat=8)]

def find_top_8mer_matches_non_overlapping(file_path):
    df = pd.read_csv(file_path)
    first_gene_sequence = df.iloc[0]['sequence']
    
    print(f"Length of the gene sequence: {len(first_gene_sequence)}")
    
    # Break the gene sequence into non-overlapping parts of 8 characters
    parts = [first_gene_sequence[i:i+8] for i in range(0, len(first_gene_sequence), 8) if len(first_gene_sequence[i:i+8]) == 8]
    
    print(f"Total number of 8-character parts: {len(parts)}")
    
    sequences = generate_8mer_sequences()
    counts = Counter(parts)             ###########remove
    
    top_10_matches = counts.most_common(10)
    print(f"Number of 'AAAAAAAA': {counts['AAAAAAAA']}")

    return top_10_matches

file_path = '/data/home/mrichte3/RNASeq/sequences.csv'
top_10_matches = find_top_8mer_matches_non_overlapping(file_path)
print(top_10_matches)


Length of the gene sequence: 12884
Total number of 8-character parts: 1610
Number of 'AAAAAAAA': 1
[('ATTTTCCT', 4), ('GAAAAAAA', 3), ('AAAAATTA', 3), ('TACAGTTA', 3), ('TTTTTTTT', 3), ('CCAGGCTG', 3), ('GAGTGAGT', 2), ('GACTTTTT', 2), ('CAATGAAA', 2), ('TTCTTGTG', 2)]


In [10]:
## FINDS MATCHES OF A SEQUENCE "AATCCTA"
import pandas as pd

sequences_file = '/data/home/mrichte3/RNASeq/sequences.csv'
alignments_file = '/data/home/mrichte3/RNASeq/gene_alignments5.csv'

sequences_df = pd.read_csv(sequences_file)
alignments_df = pd.read_csv(alignments_file)

sequences_df['count'] = sequences_df['sequence'].apply(lambda x: x.count('AATCCTA'))
sequences_df['length'] = sequences_df['sequence'].str.len()
sorted_sequences = sequences_df[['ensemble_id', 'count', 'length']].sort_values(by='count', ascending=False)
merged_df = sorted_sequences.merge(alignments_df, left_on='ensemble_id', right_on='ensembl_id', how='inner')

merged_df['log2FC_unmod'] = merged_df['log2FC_unmod'].round(3)
merged_df['matches/length'] = merged_df['count'] / merged_df['length']
merged_df['length/matches'] = merged_df['length'] / merged_df['count'].replace(0, pd.NA)

top_results = merged_df[['ensemble_id', 'count', 'length', 'log2FC_unmod', 'matches/length', 'length/matches']].head(100)
for index, row in top_results.iterrows():
    print(f"{row['ensemble_id']}, {row['count']}, matches, length, {row['length']}, log2FC_unmod, {row['log2FC_unmod']}, matches/length, {row['matches/length']}, length/matches, {row['length/matches']}")

ENSG00000198947, 194, matches, length, 2241933, log2FC_unmod, -0.24, matches/length, 8.653246997122572e-05, length/matches, 11556.355670103092
ENSG00000153707, 178, matches, length, 2298757, log2FC_unmod, 0.535, matches/length, 7.743315191644876e-05, length/matches, 12914.365168539325
ENSG00000168702, 159, matches, length, 1899594, log2FC_unmod, 0.104, matches/length, 8.370209634269217e-05, length/matches, 11947.132075471698
ENSG00000021645, 154, matches, length, 1697919, log2FC_unmod, -0.103, matches/length, 9.069926186113707e-05, length/matches, 11025.448051948051
ENSG00000144642, 143, matches, length, 1435601, log2FC_unmod, 0.269, matches/length, 9.960984981202994e-05, length/matches, 10039.167832167832
ENSG00000184305, 138, matches, length, 1477902, log2FC_unmod, -0.153, matches/length, 9.33756094788423e-05, length/matches, 10709.434782608696
ENSG00000173406, 138, matches, length, 1551957, log2FC_unmod, 0.191, matches/length, 8.8919989406923e-05, length/matches, 11246.065217391304


In [ ]:
import pandas as pd
df = pd.read_csv('raw_data_0.csv')
available_columns = [str(i) for i in range(225, 346) if i not in [245, 246, 247, 273, 274, 275]]
selected_columns = ['comp_name'] + available_columns
df_selected = df[selected_columns]
df_selected['sum_225_345'] = df_selected.iloc[:, 1:].sum(axis=1)
print(df_selected[['comp_name', 'sum_225_345']])


In [5]:
df_selected_sorted = df_selected.sort_values(by='sum_225_345', ascending=True)
print(df_selected_sorted[['comp_name', 'sum_225_345']])


                   comp_name  sum_225_345
2717   ENSG00000075234_unmod     215.4494
4118   ENSG00000092607_unmod     217.8989
7529   ENSG00000110344_unmod     218.0902
21638  ENSG00000166024_unmod     224.6727
12461  ENSG00000131459_unmod     229.3189
...                      ...          ...
2383     ENSG00000070785_gna    1799.3356
1690     ENSG00000060069_gna    1800.5160
15463    ENSG00000140961_gna    1804.7302
28799  ENSG00000198160_unmod    1812.2831
27207  ENSG00000185917_amide    1833.1625

[32307 rows x 2 columns]
